# Phase 2 · Step 4 — Time to first token (TTFT), Transformers variants

Measures how long each Transformers variant takes to return the **first** token: FP16, bitsandbytes INT8,
bitsandbytes NF4, and the two ablation variants. 30 GSM8K + 30 MMLU prompts each (the same first questions as
every other run), chat template applied, 1 generated token, GPU synchronised before and after.
llama.cpp and AWQ TTFT are measured in their own notebooks with the same 30 + 30 prompts.

**T4 GPU runtime → Runtime → Run all.** About 20–30 minutes. Results: `quant_study_phase2/runs/ttft_transformers.json`.
Each variant is saved as soon as it finishes; re-running skips finished variants.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes>=0.46.1", "accelerate", "datasets"], check=True)
import os, json, time, gc, statistics as stx, torch, transformers
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
print("transformers", transformers.__version__, "| GPU:", torch.cuda.get_device_name(0))
try:
    from google.colab import drive; drive.mount("/content/drive"); DRIVE = "/content/drive/MyDrive/quant_study_phase2"
except Exception as e:
    print("!! Drive not mounted:", e); DRIVE = "/content/no_drive"
os.makedirs(f"{DRIVE}/runs", exist_ok=True)
OUT = f"{DRIVE}/runs/ttft_transformers.json"
MODEL_ID = "microsoft/Phi-4-mini-instruct"

In [ ]:
import os, json, time, gc, torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
gsm8k = load_dataset("openai/gsm8k", "main", split="test").shuffle(seed=42).select(range(30))
mmlu = load_dataset("cais/mmlu", "all", split="test").shuffle(seed=42).select(range(30))
L = ["A", "B", "C", "D"]
GSM8K_PROMPT = ("Solve this math problem step by step, then give the final numeric answer "
                "on the last line as 'Answer: <number>'.\n\nProblem: {q}\n")
MMLU_PROMPT  = ("Answer this multiple-choice question. Respond with only the letter "
                "(A, B, C, or D) on the last line as 'Answer: <letter>'.\n\nQuestion: {q}\n")
PROMPTS = {"gsm8k": [GSM8K_PROMPT.format(q=ex["question"]) for ex in gsm8k],
           "mmlu": [MMLU_PROMPT.format(q=f"{ex['question']}\n" + "\n".join(f"{L[i]}. {c}" for i, c in enumerate(ex["choices"])))
                    for ex in mmlu]}
assert gsm8k[0]["question"].startswith("Darrell and Allen"), "subset differs from Phase 1!"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None: tok.pad_token = tok.eos_token

def nf4(skip=None):
    return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                              bnb_4bit_use_double_quant=True, llm_int8_skip_modules=skip)
CONFIGS = {
    "fp16": None,
    "bnb_8bit": BitsAndBytesConfig(load_in_8bit=True, llm_int8_threshold=6.0),
    "bnb_nf4": nf4(),
    "nf4_attn_only": nf4(["lm_head", "mlp", r".*\.mlp\..*"]),
    "nf4_mlp_only": nf4(["lm_head", "self_attn", r".*\.self_attn\..*"]),
}
print("ready")

In [ ]:
import os, json, time, gc, statistics as stx, torch
assert "CONFIGS" in globals(), "Run the cells above first: Runtime -> Run all"
res = json.load(open(OUT)) if os.path.exists(OUT) else {}
for name, qc in CONFIGS.items():
    if name in res:
        print(name, "already done:", res[name]); continue
    gc.collect(); torch.cuda.empty_cache()
    kw = dict(device_map="cuda", dtype=torch.float16)
    if qc is not None: kw["quantization_config"] = qc
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kw)
    def ttft(prompt):
        inputs = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                         return_tensors="pt", return_dict=True).to(model.device)
        torch.cuda.synchronize(); t0 = time.time()
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=1, do_sample=False, pad_token_id=tok.pad_token_id)
        torch.cuda.synchronize()
        return (time.time() - t0) * 1000, inputs["input_ids"].shape[1]
    for _ in range(2): ttft(PROMPTS["gsm8k"][0])          # warm-up (not counted)
    res[name] = {}
    for task, ps in PROMPTS.items():
        r = [ttft(p) for p in ps]
        res[name][task] = {"median_ttft_ms": round(stx.median(x for x, _ in r), 1),
                           "p90_ttft_ms": round(sorted(x for x, _ in r)[int(0.9 * len(r))], 1),
                           "median_prompt_tokens": stx.median(n for _, n in r)}
    json.dump(res, open(OUT, "w"), indent=1)
    print(name, res[name])
    del model; gc.collect(); torch.cuda.empty_cache()
print("\nSaved:", OUT)